A DataFrame is a table with labeled rows and columns. The useful skill is not memorizing every pandas method; it is learning a small grammar of operations you can combine.

## Learning goals

By the end you can:

- inspect shape, types, and representative rows;
- select columns and filter rows without ambiguity;
- create derived variables with `assign`;
- summarize groups and reshape results;
- join related tables and validate the result.

In [1]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": range(1001, 1013),
    "date": pd.to_datetime([
        "2026-08-01", "2026-08-01", "2026-08-02", "2026-08-03",
        "2026-08-03", "2026-08-04", "2026-08-05", "2026-08-05",
        "2026-08-06", "2026-08-07", "2026-08-08", "2026-08-08"
    ]),
    "region": ["North", "South", "West", "North", "West", "South",
               "North", "South", "West", "North", "South", "West"],
    "channel": ["web", "store", "web", "web", "store", "web",
                "store", "web", "web", "store", "store", "web"],
    "product_id": ["P1", "P2", "P1", "P3", "P2", "P3",
                   "P1", "P2", "P3", "P2", "P1", "P3"],
    "units": [2, 1, 4, 2, 3, 1, 5, 2, 3, 4, 2, 5],
    "unit_price": [18, 25, 18, 32, 25, 32, 18, 25, 32, 25, 18, 32],
})
orders.head()

,order_id,date,region,channel,product_id,units,unit_price
0,1001,2026-08-01,North,web,P1,2,18
1,1002,2026-08-01,South,store,P2,1,25
2,1003,2026-08-02,West,web,P1,4,18
3,1004,2026-08-03,North,web,P3,2,32
4,1005,2026-08-03,West,store,P2,3,25


## 1. Inspect before transforming

Start with four questions: How many rows are there? What does one row represent? Which columns exist? Did pandas infer sensible types? A correct analysis begins by answering those questions explicitly.

In [2]:
print(f"rows={orders.shape[0]}, columns={orders.shape[1]}")
print(orders.dtypes)
orders.sample(3, random_state=42)

rows=12, columns=7
order_id               int64
date          datetime64[us]
region                   str
channel                  str
product_id               str
units                  int64
unit_price             int64
dtype: object


,order_id,date,region,channel,product_id,units,unit_price
10,1011,2026-08-08,South,store,P1,2,18
9,1010,2026-08-07,North,store,P2,4,25
0,1001,2026-08-01,North,web,P1,2,18


## 2. Select columns and filter rows

Use `loc[row_condition, columns]` when you want the selection to be obvious to a reader. Here we ask a concrete question: which web orders in the South sold at least two units?

In [3]:
condition = (orders["region"] == "South") & (orders["channel"] == "web") & (orders["units"] >= 2)
orders.loc[condition, ["order_id", "date", "product_id", "units"]]

,order_id,date,product_id,units
7,1008,2026-08-05,P2,2


## 3. Create variables that match the question

`unit_price` is useful operationally, but total revenue is the quantity needed for regional comparison. `assign` returns a new DataFrame, which makes the transformation easy to compose.

In [4]:
enriched = orders.assign(
    revenue=lambda frame: frame["units"] * frame["unit_price"],
    weekday=lambda frame: frame["date"].dt.day_name(),
)
enriched[["order_id", "revenue", "weekday"]].head()

,order_id,revenue,weekday
0,1001,36,Saturday
1,1002,25,Saturday
2,1003,72,Sunday
3,1004,64,Monday
4,1005,75,Monday


## 4. Split, apply, combine with `groupby`

Grouping splits rows by one or more keys, applies aggregations inside each group, then combines the summaries. Name the outputs so the resulting table can stand on its own.

In [5]:
regional_summary = (
    enriched.groupby("region", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        units_sold=("units", "sum"),
        revenue=("revenue", "sum"),
        average_order=("revenue", "mean"),
    )
    .sort_values("revenue", ascending=False)
)
regional_summary.round(2)

,region,orders,units_sold,revenue,average_order
2,West,4,15,403,100.75
0,North,4,13,290,72.50
1,South,4,6,143,35.75


## 5. Reshape for comparison

A pivot table is a grouped summary displayed as a matrix. Rows and columns should represent the two dimensions you want to compare.

In [6]:
pd.pivot_table(
    enriched,
    index="region",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
)

channel,store,web
region,,
North,190,100
South,61,82
West,75,328


## 6. Join related tables safely

Product names belong in a product table, not repeated in every transactional source. The `validate` argument converts an assumption about key uniqueness into an executable check.

In [7]:
products = pd.DataFrame({
    "product_id": ["P1", "P2", "P3"],
    "product_name": ["Notebook", "Desk lamp", "Headphones"],
    "category": ["Stationery", "Home office", "Electronics"],
})

analysis_table = enriched.merge(
    products, on="product_id", how="left", validate="many_to_one"
)
assert len(analysis_table) == len(enriched)
analysis_table[["order_id", "product_name", "category", "revenue"]].head()

,order_id,product_name,category,revenue
0,1001,Notebook,Stationery,36
1,1002,Desk lamp,Home office,25
2,1003,Notebook,Stationery,72
3,1004,Headphones,Electronics,64
4,1005,Desk lamp,Home office,75


## 7. Compose a readable pipeline

Method chaining reads from top to bottom: add variables, join metadata, group, aggregate, and sort. Keep each step at one conceptual level.

In [8]:
category_report = (
    orders.assign(revenue=lambda frame: frame.units * frame.unit_price)
    .merge(products, on="product_id", validate="many_to_one")
    .groupby("category", as_index=False)
    .agg(revenue=("revenue", "sum"), units=("units", "sum"))
    .sort_values("revenue", ascending=False)
)
category_report

,category,revenue,units
0,Electronics,352,11
1,Home office,250,10
2,Stationery,234,13


## Common mistakes

- Filtering after aggregation when the question requires filtering observations first.
- Treating missing values as zeros without a domain reason.
- Joining on a non-unique key and silently multiplying rows.
- Overwriting the only copy of raw data.
- Reporting a group average without the group size.

## Try it yourself

Create a table with one row per `region` and `channel`. Include total revenue, number of orders, and revenue per order. Sort from highest to lowest revenue per order. Then answer: does the channel with the most revenue also have the largest average order?

When you are finished, continue to **[Cleaning real-world data](data-cleaning.html)**.